# GNN-BERT Music Context — Demo Notebook

Loads the trained Task 3 (early-concat fusion) model plus one saved graph + text example,
and runs a single end-to-end inference: predicting genre from a track's chroma graph + text metadata.

In [ ]:
import sys
sys.path.append('/content/gnn-bert-music-context/src')

import json
import torch
import pandas as pd
from torch_geometric.data import Batch as PyGBatch

from bert_encoder import get_tokenizer, MODEL_NAME, MAX_LEN
from fusion_model import EarlyConcatFusion

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Recover genre label ordering (matches sklearn LabelEncoder's sorted-unique order used during training)
METADATA_DIR = f"{PROJECT_DIR}/data/raw/fma_metadata"
tracks = pd.read_csv(f"{METADATA_DIR}/tracks.csv", index_col=0, header=[0, 1])
small = tracks[tracks[('set', 'subset')] == 'small']
GENRE_CLASSES = sorted(small[('track', 'genre_top')].dropna().unique())
print(GENRE_CLASSES)

In [ ]:
# Load a real worked example from the Task 3 case studies
with open(f"{PROJECT_DIR}/results/task3_case_studies.json") as f:
    case_studies = json.load(f)

example = case_studies["case_1_success"]
print(example)

In [ ]:
# Load that track's chroma segment graph from the full saved graph set
all_graphs = torch.load(f"{PROJECT_DIR}/data/graphs/all_graphs.pt")
graph_by_track = {g.track_id: g for g in all_graphs}
graph = graph_by_track[example["track_id"]]
print(graph)

In [ ]:
# Load the trained early-concat fusion model (best checkpoint from Task 3)
tokenizer = get_tokenizer()

model = EarlyConcatFusion(bert_name=MODEL_NAME).to(device)
model.load_state_dict(torch.load(f"{PROJECT_DIR}/results/task3_earlyconcat_best.pt", map_location=device))
model.eval()
print("Model loaded.")

In [ ]:
# Run one end-to-end inference: text + graph -> predicted genre
text = example["text"]
enc = tokenizer(text, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt").to(device)
graph_batch = PyGBatch.from_data_list([graph]).to(device)

with torch.no_grad():
    logits = model(graph_batch, enc["input_ids"], enc["attention_mask"])
    pred_idx = torch.argmax(logits, dim=1).item()

print(f"Text: {text}")
print(f"True genre:      {example['true_genre']}")
print(f"Predicted genre: {GENRE_CLASSES[pred_idx]}")

## Note

This demo reuses raw FMA metadata (`data/raw/fma_metadata`) only to recover the genre label ordering.
It is not required for inference itself — model weights, the graph, and the text are self-contained
in `results/` and `data/graphs/`. On a fresh clone without raw data, hardcode `GENRE_CLASSES`
using the list printed by the cell above.